In [9]:
import os 
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import START , END, StateGraph
from typing import TypedDict
import time
load_dotenv()
model = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0.7)

In [32]:
class batsmanState(TypedDict) : 
    runs : int
    balls : int
    _4s : int
    _6s : int
    strike_rate : float
    runs_per_boundary_percent : float
    balls_per_boundary  : float
    summary  :str
    

In [36]:
def calculate_sr(state: batsmanState) -> batsmanState:
    runs = state["runs"]
    balls = state["balls"]
    strike_rate = (runs / balls) * 100
    return { "strike_rate": strike_rate}

def calculate_bpb(state: batsmanState) -> batsmanState:
    total_boundaries = state["_4s"] + state["_6s"]
    balls = state["balls"]
    balls_per_boundary = balls / total_boundaries
    return {"balls_per_boundary": balls_per_boundary}

def calculate_rpb(state: batsmanState) -> batsmanState:
    runs = state["runs"]
    boundary_runs = (state["_4s"] * 4) + (state["_6s"] * 6)
    rpb = (boundary_runs / runs) * 100
    return { "runs_per_boundary_percent": rpb}

def summary(state: batsmanState) -> batsmanState:
    summary_text = f"The batsman has a strike rate of {state['strike_rate']:.2f} and runs per boundary percentage of {state['runs_per_boundary_percent']:.2f} and balls per boundary of {state['balls_per_boundary']:.2f}"
    print(summary_text)
    return { "summary": summary_text}

In [38]:
graph = StateGraph(batsmanState)
# Nodes
graph.add_node("calculate_sr", calculate_sr)
graph.add_node("calculate_bpb", calculate_bpb)
graph.add_node("calculate_rpb", calculate_rpb)
graph.add_node("summary", summary)
#Edges
graph.add_edge(START , "calculate_sr")
graph.add_edge(START , "calculate_bpb")
graph.add_edge(START , "calculate_rpb")
graph.add_edge("calculate_sr" , "summary")
graph.add_edge("calculate_bpb" , "summary")
graph.add_edge("calculate_rpb" , "summary")
graph.add_edge("summary" , END)

workflow = graph.compile()




In [39]:
initial_state = {
    "runs" : 100,
    "_4s" : 6,
    "_6s" : 5,
    "balls" : 73
}
workflow.invoke(initial_state)

The batsman has a strike rate of 136.99 and runs per boundary percentage of 54.00 and balls per boundary of 6.64


{'runs': 100,
 'balls': 73,
 '_4s': 6,
 '_6s': 5,
 'strike_rate': 136.986301369863,
 'runs_per_boundary_percent': 54.0,
 'balls_per_boundary': 6.636363636363637,
 'summary': 'The batsman has a strike rate of 136.99 and runs per boundary percentage of 54.00 and balls per boundary of 6.64'}